Load & Test Diarization Model

In [1]:
AUDIO_FILE_TEST_PATH = "/home/dzur/ai_projects/new_jh_test.wav"

In [2]:
# download the pipeline from Huggingface
from pyannote.audio import Pipeline
import torch

diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1").to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# run the pipeline locally on your computer
diarization_output = diarization_pipeline(AUDIO_FILE_TEST_PATH)

# print the predicted speaker diarization 
for turn, speaker in diarization_output.speaker_diarization:
    print(f"{speaker} speaks between t={turn.start:.3f}s and t={turn.end:.3f}s")


2026-03-07 15:50:50.852415968 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/home/dzur/ai_projects/therapy-bert/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/dzur/ai_projects/therapy-bert/.venv/lib/python3.10/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/hom

SPEAKER_01 speaks between t=0.031s and t=3.440s
SPEAKER_01 speaks between t=3.693s and t=4.925s
SPEAKER_01 speaks between t=5.195s and t=9.937s
SPEAKER_00 speaks between t=10.527s and t=14.830s
SPEAKER_00 speaks between t=15.252s and t=17.159s
SPEAKER_00 speaks between t=17.632s and t=18.087s
SPEAKER_00 speaks between t=18.425s and t=25.681s
SPEAKER_00 speaks between t=26.238s and t=28.820s
SPEAKER_00 speaks between t=29.157s and t=29.967s
SPEAKER_01 speaks between t=29.967s and t=34.085s
SPEAKER_00 speaks between t=34.237s and t=37.021s
SPEAKER_00 speaks between t=37.291s and t=41.982s
SPEAKER_00 speaks between t=42.185s and t=44.733s
SPEAKER_00 speaks between t=44.969s and t=45.543s


Dump cache before loading ASR model in

In [3]:
import gc

gc.collect()
torch.cuda.empty_cache()

Load the ASR Model

In [4]:
import torch
from faster_whisper import WhisperModel

# define our torch configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if torch.cuda.is_available() else "float32"

# load model on GPU if available, else cpu
asr_pipeline = WhisperModel("distil-whisper/distil-large-v3.5-ct2", device=device, compute_type=compute_type)

sample = AUDIO_FILE_TEST_PATH

segments, info = asr_pipeline.transcribe(sample, beam_size=5, language="en", word_timestamps=True)

# uncommenting will burn through the generator
# only uncomment for testing/experimentation!

# # 1. Loop through the main segments (the shipping containers)
# for segment in segments:
    # 2. Loop through the individual words packed inside each segment
    # for word in segment.words:
        # 3. Print the exact start, end, and text of each individual word
        # print("[%.2fs -> %.2fs] %s" % (word.start, word.end, word.word))


Build Diarized transcript

In [7]:
def build_diarized_transcript(diarization_output, whisper_segments):
    """
    Merges Pyannote speaker segments with Faster-Whisper word timestamps.
    """

    def _normalize_speaker_label(speaker_id: str) -> str:
        # bogus initialization to handle the "UNKNOWN" case
        speaker_num = 9

        if speaker_id.startswith("SPEAKER_"):
            speaker_num = int(speaker_id.split("_")[1])
            speaker_num += 1
        
        return "SPEAKER_0" + str(speaker_num)

    # 1. Map the Pyannote "parking spaces"
    speaker_turns = []
    for turn, _, speaker in diarization_output.itertracks(yield_label=True):
        # print(turn.start, turn.end, speaker)
        speaker_turns.append({
            "start": turn.start,
            "end": turn.end,
            "speaker": speaker
        })
        
    final_transcript = []
    current_speaker = None
    current_phrase = []
    
    # 2. Unpack the Whisper objects
    for segment in whisper_segments:
        for word in segment.words:
            # Find the exact middle of the word
            word_midpoint = (word.start + word.end) / 2
            assigned_speaker = "UNKNOWN"
            
            # 3. Check which speaker's block the word's midpoint falls into
            for turn in speaker_turns:
                if turn["start"] <= word_midpoint <= turn["end"]:
                    assigned_speaker = turn["speaker"]
                    break
            
            # 4. Group words back into sentences based on the speaker
            if assigned_speaker != current_speaker:
                # If the speaker changes, save the old sentence and start a new one
                if current_phrase:
                    final_transcript.append({
                        "speaker": _normalize_speaker_label(current_speaker),
                        "text": "".join(current_phrase).strip()
                    })
                current_speaker = assigned_speaker
                current_phrase = [word.word]
            else:
                # If it's the same speaker, just append the word
                current_phrase.append(word.word)
                
    # 5. Catch the very last sentence when the audio ends
    if current_phrase:
        final_transcript.append({
            "speaker": _normalize_speaker_label(current_speaker),
            "text": "".join(current_phrase).strip()
        })
        
    return final_transcript

print(build_diarized_transcript(diarization_output.speaker_diarization, segments))

[{'speaker': 'SPEAKER_02', 'text': "don't say, hey, where's your SAT score?"}, {'speaker': 'SPEAKER_01', 'text': "They weigh it differently when they're looking at applications. I don't know the specifics, but"}, {'speaker': 'SPEAKER_09', 'text': 'yeah,'}, {'speaker': 'SPEAKER_01', 'text': "you don't have to take the tests. So"}, {'speaker': 'SPEAKER_09', 'text': 'over'}, {'speaker': 'SPEAKER_01', 'text': '1,800 four-year colleges and universities in the U.S. are test optional or test flexible,'}, {'speaker': 'SPEAKER_09', 'text': 'and'}, {'speaker': 'SPEAKER_01', 'text': 'that number grew significantly during COVID. So did testing'}, {'speaker': 'SPEAKER_02', 'text': 'just move online during COVID? How did they do that? I remember how strict they were with all the proctoring.'}, {'speaker': 'SPEAKER_01', 'text': 'Remember, no one knew what the hell to do. And'}, {'speaker': 'SPEAKER_09', 'text': 'I'}, {'speaker': 'SPEAKER_01', 'text': "think standardized tests just weren't very high o